# 1. Census: what does the dataset contain? (CPU, metadata only)

Before choosing blocks, find out what is there. This downloads ONLY the small base layer of each block (1 to 3 GB),
reads road type, weather, lighting, speed and the sensor set of every scene, saves one small CSV per block to
persistent storage, and deletes the block again. No images, no LiDAR.

Resumable: blocks already counted are skipped. `MAX_BLOCKS` limits one session; `REVERSE = True` lets a second
server start from the other end of the list.

**Next:** choose `BLOCKS` from the tables in cells 5 and 6, and put the same list into `02_predict_gpu` and `03_score`.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
SPLITS = ["val", "test", "train"]     # census order. val first: it is the development split
MAX_BLOCKS = 60                       # per session. Re-run to continue; finished blocks are skipped
REVERSE = False                       # True = start from the LAST block. Run a second copy on a second server with True to halve the wait

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Count blocks, then census the ones not done yet ---
import time
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
print("blocks downloadable with camera + LiDAR:", len(available), "| scenes:", int(available["n_scenes"].sum()))

done_now = 0
todo = [(s, b) for s in SPLITS for (s2, b) in available.index if s2 == s]
for split, block in (reversed(todo) if REVERSE else todo):
    if True:
        already = pl.census_is_current(session.persist_root, split, block)   # an older table is counted again
        if already or done_now >= MAX_BLOCKS:
            continue
        started = time.time()
        table = pl.census_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED))
        done_now += 1
        print(f"{pl.block_tag(split, block)}: {len(table)} scenes in {time.time() - started:.0f} s | "
              f"{table['road_type'].value_counts().to_dict()} | lidars {sorted(table['n_lidars'].unique())}")
print("blocks counted in this session:", done_now)

scenes excluded by the dataset: 8
blocks downloadable with camera + LiDAR: 56 | scenes: 1081
$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura/census_train_block000077 --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits train --scene-ids-file /content/data/fzi-aura/census_train_block000077/_scene_ids_train_block000077.txt --layers base_keyframes --jobs 8 --verify
train_block000077: 16 scenes in 39 s | {'urban': 11, 'overland': 5} | lidars [np.int64(6), np.int64(10), np.int64(12)]
$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura/census_train_block000078 --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits train --scene-ids-file /content/data/fzi-aura/census_train_block000078/_scene_ids_train_block000078.txt --layers base_keyframes --jobs 8 --verify
train_block000078: 20 scenes in 33 s | {'urban': 18, 'overland': 2} | lidars [np.int64(6)]
$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura/census_train_block000079 --revision 3404bd6b

In [5]:
# --- 5. What has been counted so far ---
files = sorted((session.persist_root / "census").glob("[a-z]*_block[0-9]*.csv"))   # block tables only, not the _summary files
census = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()
print("blocks counted:", len(files), "of", len(available), "| scenes:", len(census))
if len(census):
    census.to_csv(session.persist_root / "census" / "_all_scenes.csv", index=False)
    for column in ("road_type", "weather_group", "lighting", "speed_band", "n_lidars", "n_lidars_ouster", "has_aeva"):
        if column not in census:
            continue
        print()
        print(f"scenes by {column}:")
        print(pd.crosstab(census["split"], census[column]).to_string())
    print()
    print("road_type x lighting x weather_group (trap 7: cells under 5 scenes cannot carry a finding):")
    print(census.groupby(["road_type", "lighting", "weather_group"]).size().rename("scenes").reset_index().to_string(index=False))

blocks counted: 56 of 56 | scenes: 1073

scenes by road_type:
road_type  motorway  overland  urban
split                               
test              1        14     92
train            76       197    586
val               2        22     83

scenes by weather_group:
weather_group  dry  wet
split                  
test            86   21
train          637  222
val             67   40

scenes by lighting:
lighting  day  night  twilight
split                         
test      107      0         0
train     766     56        37
val        93      9         5

scenes by speed_band:
speed_band  20-50 km/h  5-20 km/h  over 50 km/h  under 5 km/h
split                                                        
test                65         17             5            20
train              380        175           175           129
val                 55         17            18            17

scenes by n_lidars:
n_lidars  4    6   10   12
split                     
test       0   68   0  

In [6]:
# --- 6. Which blocks would fill the thin cells? One row per block ---
if len(census):
    per_block = census.groupby(["split", "block"]).agg(
        scenes=("scene_id", "size"), motorway=("road_type", lambda s: int((s == "motorway").sum())),
        overland=("road_type", lambda s: int((s == "overland").sum())), wet=("weather_group", lambda s: int((s == "wet").sum())),
        not_day=("lighting", lambda s: int((s != "day").sum())), fast=("speed_band", lambda s: int((s == "over 50 km/h").sum())),
        aeva=("has_aeva", "sum"), lidars=("n_lidars", "max"),
        fewest_ouster=("n_lidars_ouster", "min") if "n_lidars_ouster" in census else ("n_lidars", "min")).reset_index()
    sizes = available[["total_gb"]].reset_index().rename(columns={"scene_block": "block"})
    per_block = per_block.merge(sizes, on=["split", "block"], how="left")
    per_block["rare_scenes"] = per_block[["motorway", "wet", "not_day"]].sum(axis=1)
    print(per_block.sort_values("rare_scenes", ascending=False).to_string(index=False))
    per_block.to_csv(session.persist_root / "census" / "_per_block.csv", index=False)

split  block  scenes  motorway  overland  wet  not_day  fast  aeva  lidars  fewest_ouster  total_gb  rare_scenes
train     11      20         5        11   20       20    13    20      10              6     11.77           45
train     12      20         0        13   20       20    13    20      10              6     12.29           40
train     13      20         0        15   20       20    14    20      10              6     12.37           40
train     10      20         3         7   15       12     5    20      12              4     14.43           30
train     97      20         8         1    4        5     9     1      12              4      7.83           17
  val      1      20         0         4   12        4     3    20      12              6     16.88           16
  val      2      20         1         6   10        5     7    14      12              6     12.65           16
train     77      16         0         5   10        5     4     8      12              4     11